# Logistic Regression — Flight Delay Prediction

## Objective
Predict whether a flight arrives at least 15 minutes late.

## Methodology
- Use the shared leakage-aware data preparation pipeline and approved model-feature allowlist. Historical outcome availability at the prediction instant remains unverified.
- Train Logistic Regression on the training partition only.
- Evaluate on the existing chronological validation partition.
- Use Average Precision and ROC-AUC as evaluation metrics.
- Investigate the effect of regularization and class weighting.
- Do not use the held-out test partition for model selection.

This notebook's uncalibrated validation experiment is not the canonical frozen evaluation. Use `docs/canonical_evaluation.json` for the selected 31-feature models' validation and test metrics; the 33-feature congestion variant is validation-only.

In [1]:
import sys
from pathlib import Path

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the flight-delay repository")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from flight_delay.data.prepare import prepare_model_data
from flight_delay.models.logistic import fit_logistic_baseline
from flight_delay.features.model_features import (
    select_model_features,
)
from sklearn.metrics import average_precision_score, roc_auc_score

# Locate the dataset whether the notebook runs from the
# repository root or from the notebooks/ directory.
candidates = [
    Path.cwd() / "data" / "raw" / "flights.csv",
    Path.cwd().parent / "data" / "raw" / "flights.csv",
]

DATA_PATH = next(
    (path for path in candidates if path.is_file()),
    None,
)

if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find flights.csv. Current directory: {Path.cwd()}"
    )

print("Dataset:", DATA_PATH.resolve())

splits = prepare_model_data(DATA_PATH)

X_train, y_train = splits["train"]
X_validation, y_validation = splits["validation"]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_validation))
print("Training delay prevalence:", round(y_train.mean(), 4))
print("Validation delay prevalence:", round(y_validation.mean(), 4))

Dataset: /Users/rashmianand/Desktop/DL PROJECT/flight-delay-prediction/data/raw/flights.csv


Training rows: 228299
Validation rows: 50466
Training delay prevalence: 0.2591
Validation delay prevalence: 0.1564


In [2]:
model = fit_logistic_baseline(
    X_train,
    y_train,
    feature_set="full",
    C=1.0,
)

X_validation_selected = select_model_features(
    X_validation,
    feature_set="full",
)

validation_probabilities = model.predict_proba(
    X_validation_selected
)[:, 1]

baseline_metrics = {
    "average_precision": average_precision_score(
        y_validation, validation_probabilities
    ),
    "roc_auc": roc_auc_score(
        y_validation, validation_probabilities
    ),
}

pd.DataFrame([baseline_metrics])

,average_precision,roc_auc
0,0.203867,0.60149


In [3]:
experiment_results = []

for c_value in [0.1, 1.0, 10.0]:
    experiment_model = fit_logistic_baseline(
        X_train,
        y_train,
        feature_set="full",
        C=c_value,
    )

    probabilities = experiment_model.predict_proba(
        X_validation_selected
    )[:, 1]

    experiment_results.append({
        "C": c_value,
        "class_weight": None,
        "average_precision": average_precision_score(
            y_validation, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_validation, probabilities
        ),
    })

experiment_table = pd.DataFrame(experiment_results)
experiment_table.sort_values(
    "average_precision", ascending=False
)

,C,class_weight,average_precision,roc_auc
1,1.0,None,0.203867,0.601490
2,10.0,None,0.203767,0.601226
0,0.1,None,0.203749,0.601757


In [4]:
class_weight_results = []

for weight in [None, "balanced"]:
    experiment_model = fit_logistic_baseline(
        X_train,
        y_train,
        feature_set="full",
        C=1.0,
        class_weight=weight,
    )

    probabilities = experiment_model.predict_proba(
        X_validation_selected
    )[:, 1]

    class_weight_results.append({
        "class_weight": weight if weight else "None",
        "average_precision": average_precision_score(
            y_validation, probabilities
        ),
        "roc_auc": roc_auc_score(
            y_validation, probabilities
        ),
    })

class_weight_table = pd.DataFrame(class_weight_results)
class_weight_table

,class_weight,average_precision,roc_auc
0,None,0.203867,0.601490
1,balanced,0.206047,0.604281


## Class-Imbalance Experiment — refreshed source, cleared saved outputs

Earlier saved notebook outputs predated the shared-pipeline refactor and are
superseded. The canonical direct-comparison values are in
`docs/model_comparison.md`: unweighted Logistic Regression AP 0.203867 / ROC-AUC
0.601490 and balanced Logistic Regression AP 0.206047 / ROC-AUC 0.604281.
Rerun this notebook to regenerate local outputs. These are validation results;
the held-out test was not used for model selection.